# Dataset Overview

This notebook computes the summary statistics reported in **Table 1** of the paper:
total lineages, active vs. deleted counts, median commits per lineage,
and observed lifetime (in days).

## Definitions

| Term | Definition |
|------|------------|
| **Lineage** | A canonical detection rule tracked across its full commit history. |
| **Active** | `exists_in_head == True` — present in the HEAD of the repository at snapshot date. |
| **Deleted** | `exists_in_head == False` — removed from the repository before the snapshot. |
| **Observed lifetime** | Days from `first_commit_date` to either (a) `deleted_date` if deleted, or (b) the repository snapshot date if still active. This is a right-censored measurement for active lineages. |
| **Commits** | Total number of commits that touch the lineage file(s), i.e. `commit_count` from the lineage metadata. |

## Data sources

| File | Role |
|------|------|
| `build_data/lineage_metadata_final_{repo}.json` | Per-lineage metadata: active/deleted flag, dates, commit count. |
| `build_data/rule_versions_{repo}.jsonl` | Defines the analysis-ready subset: only lineages that yielded at least one parseable SPL version appear here. |

The analysis-ready subset is the intersection:
lineages that appear in `rule_versions_{repo}.jsonl`
cross-joined with their metadata from `lineage_metadata_final_{repo}.json`.

In [10]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Adjust these paths if running from a location other than analysis/scripts/.
# SNAPSHOT_DATE: the date the repository was pulled; used as the right-censoring
# endpoint for active lineages.

from pathlib import Path
from datetime import datetime, timezone

# Root of the repository
REPO_ROOT = Path("../..").resolve()
BUILD_DATA = REPO_ROOT / "data_prep" / "build_data"

# Repository snapshot date (used as right-censoring endpoint for active lineages)
SNAPSHOT_DATE = datetime(2026, 4, 10, tzinfo=timezone.utc)

# Dataset labels used in the output table
DATASETS = {
    "Sigma":     "sigma",
    "Splunk SC": "ssc",
}

print(f"REPO_ROOT    : {REPO_ROOT}")
print(f"BUILD_DATA   : {BUILD_DATA}")
print(f"SNAPSHOT_DATE: {SNAPSHOT_DATE.date()}")

REPO_ROOT    : /Users/elena/Desktop/Evolution-of-Log-Based-Detection-Rules
BUILD_DATA   : /Users/elena/Desktop/Evolution-of-Log-Based-Detection-Rules/data_prep/build_data
SNAPSHOT_DATE: 2026-04-10


## Helper functions

- `parse_utc(s)` — parse ISO-8601 timestamp strings (with or without `Z` suffix) to UTC `datetime`.
- `load_analysis_lineages(repo)` — load and filter metadata to the analysis-ready subset
  (lineages that appear in `rule_versions_{repo}.jsonl`).
- `compute_dataset_stats(records, snapshot)` — compute all summary statistics for one dataset.

In [14]:
import json
import statistics
from typing import Any, Dict, List, Optional


def parse_utc(s: Optional[str]) -> Optional[datetime]:
    """Parse an ISO-8601 timestamp string to a UTC-aware datetime.
    Returns None if the input is None, empty, or unparseable.
    """
    if not s:
        return None
    s = str(s).strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        dt = datetime.fromisoformat(s)
    except ValueError:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def load_analysis_lineages(repo: str) -> List[Dict[str, Any]]:
    """Load lineage metadata for the analysis-ready subset of *repo*.

    Strategy:
      1. Collect all lineage_ids that appear in rule_versions_{repo}.jsonl.
         These are the lineages that yielded at least one parseable SPL version.
      2. Filter lineage_metadata_final_{repo}.json to that subset.

    Returns a list of metadata dicts.
    """
    # Step 1 — identify analysis-ready lineage IDs
    rv_path = BUILD_DATA / f"rule_versions_{repo}.jsonl"
    analysis_ids: set = set()
    with rv_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                analysis_ids.add(json.loads(line)["lineage_id"])

    # Step 2 — load and filter metadata
    meta_path = BUILD_DATA / f"lineage_metadata_final_{repo}.json"
    with meta_path.open(encoding="utf-8") as f:
        all_meta = json.load(f)

    return [m for m in all_meta if m["lineage_id"] in analysis_ids]


def compute_dataset_stats(
    records: List[Dict[str, Any]],
    snapshot: datetime,
) -> Dict[str, Any]:
    """Compute summary statistics for a list of lineage metadata records.

    Returns a dict with:
      total, active, deleted,
      total_commits, median_commits,
      median_lifetime_days, max_lifetime_days
    """
    n_active  = sum(1 for r in records if     r.get("exists_in_head"))
    n_deleted = sum(1 for r in records if not r.get("exists_in_head"))

    # Commit counts
    commit_counts = []
    for r in records:
        cc = r.get("commit_count")
        if cc is not None:
            commit_counts.append(int(cc))

    # Observed lifetimes (right-censored at snapshot for active lineages)
    lifetimes_days = []
    for r in records:
        first = parse_utc(r.get("first_commit_date"))
        if first is None:
            continue
        if r.get("exists_in_head"):
            end = snapshot                            # right-censored
        else:
            end = parse_utc(r.get("deleted_date"))   # exact end
            if end is None:
                continue
        lifetimes_days.append((end - first).total_seconds() / 86_400.0)

    return {
        "total":                len(records),
        "active":               n_active,
        "deleted":              n_deleted,
        "total_commits":        sum(commit_counts),
        "median_commits":       statistics.median(commit_counts) if commit_counts else None,
        "median_lifetime_days": statistics.median(lifetimes_days) if lifetimes_days else None,
        "max_lifetime_days":    max(lifetimes_days)               if lifetimes_days else None,
    }


print("Helpers defined.")

Helpers defined.


## Table 1: Rule lineages studied

For active lineages, lifetime is computed up to the repository snapshot date.

In [15]:
import pandas as pd


def fmt_int(x) -> str:
    """Format an integer with thousands separator."""
    return f"{int(x):,}" if x is not None else "—"


def fmt_median_commits(x) -> str:
    """Format median commits, preserving halves if needed."""
    if x is None:
        return "—"
    if float(x).is_integer():
        return f"{int(x):,}"
    return f"{x:,.1f}"


def fmt_days(x) -> str:
    """Format a float as whole days for the paper table."""
    return f"{round(x):.0f} days" if x is not None else "—"


results: Dict[str, Dict[str, Any]] = {}
rows = []

for label, repo in DATASETS.items():
    records = load_analysis_lineages(repo)
    stats = compute_dataset_stats(records, SNAPSHOT_DATE)
    results[label] = stats
    rows.append({
        "Dataset": label,
        "Total": fmt_int(stats["total"]),
        "Active": fmt_int(stats["active"]),
        "Deleted": fmt_int(stats["deleted"]),
        "Commits": fmt_int(stats["total_commits"]),
        "Median Commits": fmt_median_commits(stats["median_commits"]),
        "Max Lifetime": fmt_days(stats["max_lifetime_days"]),
    })

df = pd.DataFrame(rows).set_index("Dataset")
display(df)

,Total,Active,Deleted,Commits,Median Commits,Max Lifetime
Dataset,,,,,,
Sigma,"4,204","3,921",283,"44,565",9,3394 days
Splunk SC,"2,655","2,310",345,"110,763",39,2669 days
